# 05 Static Phantom dataset and transfer to the Phantom flow dataset

Static HeLa and NIH3T3 cells imaged with the Phantom TMX (0.925 µm/px) in bright field (BF) and Ph2. Both cell lines
were recorded on each of the two acquisition days.

1. **Grouped cross-validation** (five folds). A group joins all crops with the same raw-image file name, cell line and
   day, also across BF and Ph2 (119 groups). BF and Ph2 are trained together and evaluated together and per mode.
2. **Held-out acquisition day:** trained on one day, tested on the other.
3. **Transfer to the Phantom flow dataset:** trained on all static Phantom images (BF and Ph2), tested on all Phantom
   flow images. The flow labels are used for the evaluation only.

Models: chip network (8 × 8) and CNN (64 × 64), three seeds each, and the feature control.

Thesis: Results, Sections "Classification of the static Phantom dataset" (Table `tab:res-static-phantom`) and
"Classification of the Phantom flow dataset", paragraph "Transfer from the static Phantom dataset" (Table
`tab:res-transfer-phantom`).

Output: `results/05_static_phantom_cv_*`, `05_static_phantom_day_*`, `05_static_phantom_to_flow_*`.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
from IPython.display import display

import thesis_config as C     # paths and settings
import thesis_ml as M         # data, preprocessing, models, training and evaluation

pd.set_option('display.precision', 3)
C.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('Results folder:', C.RESULTS_DIR.resolve())
if C.FAST:
    print('QUICK TEST (THESIS_FAST=1): few images and 3 epochs, the numbers are not meaningful.')

## 1. Data

In [ ]:
data = C.load_data(sets=['phantom_static', 'flow_phantom'])
PS, PF = data['phantom_static'], data['flow_phantom']
G = C.PS_GROUP_COL
print('Groups per class:', PS.meta.groupby('label')[G].nunique().to_dict())
display(PS.meta.groupby(['date', 'label', 'mode']).size().unstack('mode'))

## 2. Grouped cross-validation (BF and Ph2 together)

In [ ]:
cv_m, cv_p = [], []
for kind, size in C.MODELS:
    m, p, _ = M.run_grouped_cv(PS.meta, PS.X, kind, size, G, C.TRAIN, C.N_FOLDS, C.SEEDS_BY_MODEL[kind],
                               C.VAL_FRAC, scenario='ps_cv')
    cv_m.append(m); cv_p.append(p)
cv_m.append(M.run_control_cv(PS.meta, PS.feat, G, C.N_FOLDS, scenario='ps_cv'))
cv_metrics, cv_preds = pd.concat(cv_m, ignore_index=True), pd.concat(cv_p, ignore_index=True)
M.save_results('05_static_phantom_cv', C.RESULTS_DIR, cv_metrics, cv_preds)
display(cv_metrics.groupby('model').macro_f1.agg(['mean', 'std', 'count']).round(3))

Macro-F1 per imaging mode: the same networks (trained on BF and Ph2 together), evaluated separately on the BF and the
Ph2 test images of every fold. Mean over folds and seeds.

In [ ]:
RN = {'recall_0': f'recall {C.CLASSES[0]}', 'recall_1': f'recall {C.CLASSES[1]}'}
by_mode = M.grouped_metrics(cv_preds, ['mode', 'fold'])
display(by_mode.groupby(['model', 'mode'])[['macro_f1', 'recall_0', 'recall_1']].mean().rename(columns=RN).round(3))

## 3. Held-out acquisition day
Each fold trains on one day and tests on the other. Validation within the training day uses the same groups as above.

In [ ]:
days = sorted(PS.meta.date.unique())
assert all(PS.meta.loc[PS.meta.date.eq(d), 'label'].nunique() == len(C.CLASSES) for d in days)
folds = [(np.flatnonzero(PS.meta.date.ne(d)), np.flatnonzero(PS.meta.date.eq(d))) for d in days]
names = [f'test {d}' for d in days]
day_m, day_p = [], []
for kind, size in C.MODELS:
    m, p, _ = M.run_grouped_cv(PS.meta, PS.X, kind, size, 'date', C.TRAIN, None, C.SEEDS_BY_MODEL[kind], C.VAL_FRAC,
                               scenario='ps_cv_day', val_group_col=G, folds=folds, fold_names=names)
    day_m.append(m); day_p.append(p)
day_m.append(M.run_control_cv(PS.meta, PS.feat, 'date', None, scenario='ps_cv_day', folds=folds, fold_names=names))
day_metrics, day_preds = pd.concat(day_m, ignore_index=True), pd.concat(day_p, ignore_index=True)
M.save_results('05_static_phantom_day', C.RESULTS_DIR, day_metrics, day_preds)
display(day_metrics.groupby(['model', 'fold_name'])[['macro_f1', 'recall_0', 'recall_1']].mean()
        .rename(columns=RN).round(3))

## 4. Transfer to the Phantom flow dataset
Trained on all static Phantom images (epoch selection on validation groups of the static images), tested on all
Phantom flow images. Macro-F1: mean ± standard deviation over the seeds.

In [ ]:
targets = {'flow_phantom': (PF.meta, PF.X)}
tr_m, tr_p = [], []
for kind, size in C.MODELS:
    m, p = M.run_transfer(PS.meta, PS.X, targets, kind, size, G, C.TRAIN, C.SEEDS_BY_MODEL[kind], C.VAL_FRAC,
                          scenario='train_phantom_static')
    tr_m.append(m); tr_p.append(p)
tr_m.append(M.control_transfer(PS.meta, PS.feat, {'flow_phantom': (PF.meta, PF.feat)}, G,
                               scenario='train_phantom_static'))
tr_metrics, tr_preds = pd.concat(tr_m, ignore_index=True), pd.concat(tr_p, ignore_index=True)
M.save_results('05_static_phantom_to_flow', C.RESULTS_DIR, tr_metrics, tr_preds)
display(tr_metrics.groupby('model')[['macro_f1', 'recall_0', 'recall_1']].agg(['mean', 'std'])
        .rename(columns=RN).round(3))